# ElectInfo assembles a Q1 PDF deliverable for Acme Campaign

## What this shows
End-to-end report assembly: one chart, one figure caption, one table, and one `Argument` from the survey wave (`reports/03_polling_survey_analysis`) pulled together into a single branded PDF under ElectInfo's colors. Uses `ChartGenerator` for the plot and `ReportGenerator` for the PDF flow.

## Why it matters
This is the last-mile surface between analysis and client delivery. The same `Argument`-based inputs feed both the PDF path here and the slides path in `reports/02_slides_pptx_and_google.ipynb` — one upstream, two downstream.

## Prereqs
- `pip install 'siege-utilities[reporting]'` (reportlab + matplotlib).
- No credentials.

## Next
- `reports/02_slides_pptx_and_google.ipynb` — same Arguments, deck surface.
- `analytics/02_ga_end_to_end.ipynb` — Masai's weekly GA digest in the same PDF style.


## 1. Build a chart image under ElectInfo branding

ReportGenerator consumes `reportlab.platypus.Image` flowables for its chart sections. The easiest route is: render with matplotlib to a file, then load as a ReportLab Image. We use the `elect_info` branding template's primary color so the chart matches the rest of the deliverable.

In [1]:
%matplotlib inline
from pathlib import Path
import matplotlib.pyplot as plt
from siege_utilities.reporting.client_branding import ClientBrandingManager

brand = ClientBrandingManager().branding_templates['elect_info']
out = Path('output'); out.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    ['Jan', 'Feb', 'Mar'], [42, 45, 51],
    marker='o', color=brand['colors']['primary'], linewidth=2,
)
ax.set_title('Acme Q1 — share of volunteer sign-ups (%)',
             color=brand['colors']['primary'])
ax.grid(True, alpha=0.3)
fig.tight_layout()
chart_path = out / 'acme_q1_signups.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'chart written to {chart_path}')


chart written to output/acme_q1_signups.png


## 2. Build a summary table

Small pandas DataFrames flow straight into the ReportGenerator table section. No special conversion needed — `add_table_section` takes a list of dicts or a DataFrame.

In [2]:
import pandas as pd

perf_table = pd.DataFrame([
    {'metric': 'volunteer sign-ups', 'Q1 2026': 51, 'Q4 2025 baseline': 37, 'delta': '+14'},
    {'metric': 'low-dollar donors',  'Q1 2026': 1820, 'Q4 2025 baseline': 1400, 'delta': '+420'},
    {'metric': 'small-event RSVPs',  'Q1 2026': 640, 'Q4 2025 baseline': 510, 'delta': '+130'},
])
perf_table


,metric,Q1 2026,Q4 2025 baseline,delta
0,volunteer sign-ups,51,37,+14
1,low-dollar donors,1820,1400,+420
2,small-event RSVPs,640,510,+130


## 3. Assemble the PDF with ReportGenerator

`ReportGenerator` takes a client name, a set of flowables (chart, table, narrative text), and emits a branded PDF. We use the `comprehensive_report` builder because it applies ElectInfo's header / footer / page numbering automatically from the branding template.

In [3]:
from siege_utilities.reporting.report_generator import ReportGenerator

rg = ReportGenerator(client_name='elect_info', output_dir=out)
report = rg.create_comprehensive_report(
    title='TX-32: Q1 2026 engagement snapshot',
    author='ElectInfo — Acme Campaign engagement',
)
# Add sections in narrative order
rg.add_text_section(
    report, 'Summary',
    'Volunteer sign-ups up 38% versus baseline; low-dollar donor base expanded meaningfully.',
)
rg.add_chart_section(
    report, 'Sign-up trajectory', charts=[str(chart_path)],
    description='Monthly volunteer sign-ups as a share of registered Democrats in TX-32.',
)
rg.add_table_section(
    report, 'Q1 performance table', table_data=perf_table.to_dict('records'),
)
print('report sections prepared :', sorted(report.keys()))


report sections prepared : ['document_structure', 'metadata', 'sections', 'title', 'type']


## 4. Write the PDF

`generate_pdf_report` serializes the composed report. The output is a real PDF under `output/` (ignored by git). In production, ElectInfo would email the resulting PDF to the campaign's comms lead.

In [4]:
pdf_path = out / 'acme_q1_2026_engagement.pdf'
_ok = rg.generate_pdf_report(report, output_path=str(pdf_path))
assert _ok, 'PDF generation failed'
print(f'PDF written: {pdf_path}')
print(f'size: {Path(pdf_path).stat().st_size:,} bytes')


[siege_utilities] 2026-04-24 17:34:29,578 WARNING: Liberation-Serif not fully registered. Defaulting to standard ReportLab fonts.


[siege_utilities] 2026-04-24 17:34:29,582 INFO: Image cache directory ensured: /Users/dheerajchand/.siege_utilities/image_cache


[siege_utilities] 2026-04-24 17:34:29,605 INFO: Document 'output/acme_q1_2026_engagement.pdf' built successfully.


[siege_utilities] 2026-04-24 17:34:29,606 INFO: PDF report generated successfully: output/acme_q1_2026_engagement.pdf


PDF written: output/acme_q1_2026_engagement.pdf
size: 43,583 bytes


## Related

- **Source**: `siege_utilities/reporting/report_generator.py`, `siege_utilities/reporting/chart_generator.py`, `siege_utilities/reporting/client_branding.py`.
- **Tests**: `tests/test_chart_types*.py`, `tests/test_client_branding*.py`, `tests/test_reporting_config_exports.py`.
- **Next**: `reports/02_slides_pptx_and_google.ipynb` takes the same Arguments and renders them as a branded deck for Masai Interactive's client.
